# Notebook 6 – Interaction Features

Feature interaction happens when the combined effect of two or more inputs on a model's prediction is different from the sum of their separate effects

## When Interaction Features Are Useful
Interaction features combine two or more existing features to reveal relationships that neither feature shows alone — e.g. Quantity × Price reveals total transaction value, which neither `Quantity` nor `Price` alone captures. They're especially useful for **linear models** (Linear/Logistic Regression) which can't naturally detect combined effects between features, since these models assume each feature acts independently.

In [1]:
import pandas as pd
df = pd.read_csv('data.csv', encoding='ISO-8859-1')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## 1. What Are Interaction Features?

**Explanation:** Interaction features are new columns created by combining two (or more) existing features using math operations (×, +, −, /) to capture a relationship between them.

**Example:** `Quantity × UnitPrice` = Total value of a purchase line — a relationship neither column shows alone.

**Why we use / need:** Some patterns only emerge when features are combined — a model can't learn "6 items at $2.55 = $15.30" unless that relationship is explicitly given as a feature (for linear models).

**When to use:** Use when you suspect two features together explain the target better than either alone.

In [2]:
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df[['Quantity', 'UnitPrice', 'TotalPrice']].head()

,Quantity,UnitPrice,TotalPrice
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


**Code Explanation:** Multiplies `Quantity` and `UnitPrice` row-wise to get the total value of each transaction line.

## 2. Feature × Feature (Multiplication)

**Explanation:** Multiplies two features together, often used when their combined effect matters more than either alone.

**Example:** Income × Age → captures spending power that grows with both income and age together.

**Why we use / need:** Multiplication captures **compounding effects** — e.g. a high-income older person may spend very differently than a high-income young person.

**When to use:** Use when the combined magnitude of two features drives the outcome, like revenue = quantity × price.

In [3]:
df['Quantity_x_UnitPrice'] = df['Quantity'] * df['UnitPrice']
df[['Quantity', 'UnitPrice', 'Quantity_x_UnitPrice']].head()

,Quantity,UnitPrice,Quantity_x_UnitPrice
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


**Code Explanation:** Creates a new column by multiplying Quantity and UnitPrice for each row.

## 3. Feature + Feature (Addition)

**Explanation:** Adds two features together to represent a combined total.

**Example:** Product Price + Shipping Cost = Total Cost to customer

**Why:** Useful when two separate cost or value components should be viewed as one combined total for the model.

**When:** Use when features represent parts of a whole that should be summed, e.g. base price + tax.

In [4]:
df['UnitPrice_plus_Quantity'] = df['UnitPrice'] + df['Quantity']
df[['UnitPrice', 'Quantity', 'UnitPrice_plus_Quantity']].head()

,UnitPrice,Quantity,UnitPrice_plus_Quantity
0,2.55,6,8.55
1,3.39,6,9.39
2,2.75,8,10.75
3,3.39,6,9.39
4,3.39,6,9.39


**Code Explanation:** Adds UnitPrice and Quantity row-wise — shown here for demonstration, though in practice addition should only be used when it's meaningful (e.g. combining costs, not unrelated units).

## 4. Feature − Feature (Subtraction)

**Explanation:** Subtracts one feature from another to capture a difference or gap.

**Example:** Expected Price − Actual Price = Discount Amount

**Why we use / need:** Differences often reveal deviations, gaps, or changes that raw values alone don't show — like discount size or delay in days.

**When to use:** Use when the gap between two values matters, e.g. planned vs actual, expected vs received.

In [5]:
avg_price = df['UnitPrice'].mean()
df['PriceDeviation'] = df['UnitPrice'] - avg_price
df[['UnitPrice', 'PriceDeviation']].head()

,UnitPrice,PriceDeviation
0,2.55,-2.061114
1,3.39,-1.221114
2,2.75,-1.861114
3,3.39,-1.221114
4,3.39,-1.221114


**Code Explanation:** Subtracts the overall average UnitPrice from each row's price to show how much it deviates from the average.

## 5. Feature / Feature (Ratio / Division)

**Explanation:** Divides one feature by another to create a ratio or rate, normalizing one value relative to another.

**Example:** Experience / Age → shows what fraction of someone's life was spent gaining experience.

**Why we use / need:** Ratios remove scale differences and reveal relative relationships — e.g. spending per order is more meaningful than raw total spending alone.

**When to use:** Use when comparing two related quantities matters more than their absolute values, e.g. rate, average, efficiency.

In [6]:
customer_agg = df.groupby('CustomerID').agg(
    TotalSpending=('TotalPrice', 'sum'),
    NumOrders=('InvoiceNo', 'nunique')
).reset_index()
customer_agg['AvgSpendingPerOrder'] = customer_agg['TotalSpending'] / customer_agg['NumOrders']
customer_agg.head()

,CustomerID,TotalSpending,NumOrders,AvgSpendingPerOrder
0,12346.0,0.00,2,0.000000
1,12347.0,4310.00,7,615.714286
2,12348.0,1797.24,4,449.310000
3,12349.0,1757.55,1,1757.550000
4,12350.0,334.40,1,334.400000


**Code Explanation:** Divides TotalSpending by NumOrders per customer to get average spending per order — a ratio feature.

## 6. Polynomial Features

**Explanation:** Raises a feature to a power (e.g. squared) to capture non-linear relationships.

**Example:** Quantity²→ captures accelerating effects, like bulk discounts kicking in faster at higher quantities.

**Why:** Linear models can't detect curved (non-linear) relationships on their own — polynomial terms let them approximate curves.

**When:** Use when you suspect the relationship between a feature and the target isn't a straight line, e.g. diminishing or accelerating returns.

In [7]:
df['Quantity_Squared'] = df['Quantity'] ** 2
df[['Quantity', 'Quantity_Squared']].head()

,Quantity,Quantity_Squared
0,6,36
1,6,36
2,8,64
3,6,36
4,6,36


**Code Explanation:** ** 2 squares the Quantity column, creating a polynomial (non-linear) version of the feature.

## 7. Pairwise Interactions

**Explanation:** Systematically creates interaction terms for multiple pairs of features at once, rather than one at a time manually.

**Example:** From [Quantity, UnitPrice, Year] → creates Quantity×UnitPrice, Quantity×Year, UnitPrice×Year

**Why we use / need:** Useful for quickly exploring many possible combined effects when you're not sure which pairs matter — but should be filtered afterward to avoid unnecessary complexity.

**When to use:** Use during feature exploration/experimentation, not as a default for production models with many features.

In [8]:
from itertools import combinations
num_cols = ['Quantity', 'UnitPrice']
for col1, col2 in combinations(num_cols, 2):
    df[f'{col1}_x_{col2}'] = df[col1] * df[col2]
df[['Quantity', 'UnitPrice', 'Quantity_x_UnitPrice']].head()

,Quantity,UnitPrice,Quantity_x_UnitPrice
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


**Code Explanation:** combinations(num_cols, 2) generates all unique pairs from the column list; the loop multiplies each pair and stores it as a new column.

## 8. Domain-Based Interactions

**Explanation:** Interaction features built using business/domain knowledge rather than blindly combining all features — the most reliable and interpretable type.

**Example:** Total Spending / Number of Orders = Average Order Value — a well-known retail metric.

**Why we use / need:** Domain-driven interactions are meaningful and explainable to business stakeholders, unlike random combinations which may just be statistical noise.

**When to use:** Use whenever you understand the business context — always prefer domain-based interactions over blind pairwise generation.

In [9]:
customer_summary = df.groupby('CustomerID').agg(
    TotalSpending=('TotalPrice', 'sum'),
    NumOrders=('InvoiceNo', 'nunique'),
    TotalQuantity=('Quantity', 'sum')
).reset_index()
customer_summary['AvgOrderValue'] = customer_summary['TotalSpending'] / customer_summary['NumOrders']
customer_summary['AvgItemsPerOrder'] = customer_summary['TotalQuantity'] / customer_summary['NumOrders']
customer_summary.head()

,CustomerID,TotalSpending,NumOrders,TotalQuantity,AvgOrderValue,AvgItemsPerOrder
0,12346.0,0.00,2,0,0.000000,0.000000
1,12347.0,4310.00,7,2458,615.714286,351.142857
2,12348.0,1797.24,4,2341,449.310000,585.250000
3,12349.0,1757.55,1,631,1757.550000,631.000000
4,12350.0,334.40,1,197,334.400000,197.000000


**Code Explanation:** Builds two domain-meaningful ratios — AvgOrderValue (spending efficiency) and AvgItemsPerOrder (basket size) — both directly interpretable in a business context.